# OTTO — cuDF Candidate Gen + **XGBRanker(group)** Rerank + **Radek CV** (RULE / RANK / BLEND)

**What you get in this notebook**
- GPU **cuDF** co-vis matrices (clicks / carts_orders / buy2buy)
- **XGBRanker** with `group` (per-session candidates), GPU hist + GPU predictor
- A unified **batch inference** with optional **BLEND** (alpha * model + (1-alpha) * rule)
- Fast **Radek CV** (last 24h) implemented with cuDF to compute Recall@20 for RULE / RANK / BLEND

**Key fixes vs your last version**
- `top_clicks/top_orders` computed on numeric `type` (0/1/2) — no strings
- Feature `w_b2b` now correctly uses **agg_b2b**
- Ranker replaces classifier; supports **BLEND**
- CV cell prints the three curves in one go


In [ ]:
import os, gc, glob, itertools, pickle, math, random, warnings
import numpy as np, pandas as pd
from collections import Counter, defaultdict
import cudf
print('RAPIDS cuDF version:', cudf.__version__)
warnings.filterwarnings('ignore')


## Step 1 — Build three co-visitation matrices with cuDF (GPU)

In [ ]:
VER = 6  # version tag for output parquets
READ_CT = 5
type_labels = {'clicks':0, 'carts':1, 'orders':2}

# Read parquet into CPU cache (pandas) once; convert to cuDF only when needed
def _read_file_to_cache(f):
    df = pd.read_parquet(f)
    # unify ts into seconds int32
    if df['ts'].max() > 1e10:
        df['ts'] = (df['ts']/1000).astype('int32')
    else:
        df['ts'] = df['ts'].astype('int32')
    if df['type'].dtype=='O':
        df['type'] = df['type'].map(type_labels).astype('int8')
    return df

data_cache = {}
files = sorted(glob.glob('../input/otto-chunk-data-inparquet-format/*_parquet/*'))
for f in files:
    data_cache[f] = _read_file_to_cache(f)

CHUNK = int(np.ceil(len(files)/6))
print(f'Cached {len(files)} parquet files. Outer chunk size = {CHUNK}.')

def _read_gpu(flist):
    return cudf.DataFrame( pd.concat([data_cache[f] for f in flist], ignore_index=True) )


In [ ]:
# === 1) carts_orders co-vis (type weighted) ===
type_weight = {0:1.0, 1:6.0, 2:3.0}
DISK_PIECES = 4
SIZE = 1.86e6/ DISK_PIECES

for PART in range(DISK_PIECES):
    print(f"\n### DISK PART {PART+1}/{DISK_PIECES}")
    tmp = None
    for j in range(6):
        a, b = j*CHUNK, min((j+1)*CHUNK, len(files))
        print(f'Processing files {a}~{b-1} in groups of {READ_CT} ...')
        tmp2 = None
        for k in range(a, b, READ_CT):
            batch_files = [files[x] for x in range(k, min(k+READ_CT, b))]
            df = _read_gpu(batch_files)
            df = df.sort_values(['session','ts'], ascending=[True, False])
            df = df.reset_index(drop=True)
            df['n'] = df.groupby('session').cumcount()
            df = df.loc[df['n']<30].drop(columns=['n'])
            df = df.merge(df, on='session')
            df = df.loc[( (df['ts_x']-df['ts_y']).abs() < 24*60*60 ) & (df['aid_x']!=df['aid_y'])]
            df = df.loc[(df['aid_x']>=PART*SIZE) & (df['aid_x']< (PART+1)*SIZE)]
            df = df[['session','aid_x','aid_y','type_y']].drop_duplicates(['session','aid_x','aid_y'])
            df['wgt'] = df['type_y'].map(type_weight).astype('float32')
            df = df[['aid_x','aid_y','wgt']].groupby(['aid_x','aid_y']).wgt.sum()
            tmp2 = df if tmp2 is None else tmp2.add(df, fill_value=0)
            print(k,', ', end='')
        print()
        tmp = tmp2 if tmp is None else tmp.add(tmp2, fill_value=0)
        del tmp2; gc.collect()
    tmp = tmp.reset_index().sort_values(['aid_x','wgt'], ascending=[True, False])
    tmp = tmp.reset_index(drop=True)
    tmp['n'] = tmp.groupby('aid_x').aid_y.cumcount()
    tmp = tmp.loc[tmp['n']<15].drop(columns=['n'])
    tmp.to_pandas().to_parquet(f'top_15_carts_orders_v{VER}_{PART}.pqt')
print('carts_orders done.')


In [ ]:
# === 2) buy2buy co-vis ===
DISK_PIECES = 1
SIZE = 1.86e6/ DISK_PIECES
for PART in range(DISK_PIECES):
    print(f"\n### DISK PART {PART+1}/{DISK_PIECES}")
    tmp = None
    for j in range(6):
        a, b = j*CHUNK, min((j+1)*CHUNK, len(files))
        print(f'Processing files {a}~{b-1} in groups of {READ_CT} ...')
        tmp2 = None
        for k in range(a, b, READ_CT):
            batch_files = [files[x] for x in range(k, min(k+READ_CT, b))]
            df = _read_gpu(batch_files)
            df = df[df['type'].isin([1,2])]
            df = df.sort_values(['session','ts'], ascending=[True, False])
            df = df.reset_index(drop=True)
            df['n'] = df.groupby('session').cumcount()
            df = df.loc[df['n']<30].drop(columns=['n'])
            df = df.merge(df, on='session')
            df = df.loc[( (df['ts_x']-df['ts_y']).abs() < 14*24*60*60 ) & (df['aid_x']!=df['aid_y'])]
            df = df.loc[(df['aid_x']>=PART*SIZE) & (df['aid_x']< (PART+1)*SIZE)]
            df = df[['session','aid_x','aid_y']].drop_duplicates(['session','aid_x','aid_y'])
            df['wgt'] = 1.0
            df = df[['aid_x','aid_y','wgt']].groupby(['aid_x','aid_y']).wgt.sum()
            tmp2 = df if tmp2 is None else tmp2.add(df, fill_value=0)
            print(k,', ', end='')
        print()
        tmp = tmp2 if tmp is None else tmp.add(tmp2, fill_value=0)
        del tmp2; gc.collect()
    tmp = tmp.reset_index().sort_values(['aid_x','wgt'], ascending=[True, False])
    tmp = tmp.reset_index(drop=True)
    tmp['n'] = tmp.groupby('aid_x').aid_y.cumcount()
    tmp = tmp.loc[tmp['n']<15].drop(columns=['n'])
    tmp.to_pandas().to_parquet(f'top_15_buy2buy_v{VER}_{PART}.pqt')
print('buy2buy done.')


In [ ]:
# === 3) clicks co-vis (time weighted) ===
DISK_PIECES = 4
SIZE = 1.86e6/ DISK_PIECES
# constants for time weight
T0, T1 = 1659304800, 1662328791  # from public baseline
for PART in range(DISK_PIECES):
    print(f"\n### DISK PART {PART+1}/{DISK_PIECES}")
    tmp = None
    for j in range(6):
        a, b = j*CHUNK, min((j+1)*CHUNK, len(files))
        print(f'Processing files {a}~{b-1} in groups of {READ_CT} ...')
        tmp2 = None
        for k in range(a, b, READ_CT):
            batch_files = [files[x] for x in range(k, min(k+READ_CT, b))]
            df = _read_gpu(batch_files)
            df = df.sort_values(['session','ts'], ascending=[True, False])
            df = df.reset_index(drop=True)
            df['n'] = df.groupby('session').cumcount()
            df = df.loc[df['n']<30].drop(columns=['n'])
            df = df.merge(df, on='session')
            df = df.loc[( (df['ts_x']-df['ts_y']).abs() < 24*60*60 ) & (df['aid_x']!=df['aid_y'])]
            df = df.loc[(df['aid_x']>=PART*SIZE) & (df['aid_x']< (PART+1)*SIZE)]
            df = df[['session','aid_x','aid_y','ts_x']].drop_duplicates(['session','aid_x','aid_y'])
            df['wgt'] = 1.0 + 3.0*(df['ts_x']-T0)/(T1-T0)
            df = df[['aid_x','aid_y','wgt']].groupby(['aid_x','aid_y']).wgt.sum()
            tmp2 = df if tmp2 is None else tmp2.add(df, fill_value=0)
            print(k,', ', end='')
        print()
        tmp = tmp2 if tmp is None else tmp.add(tmp2, fill_value=0)
        del tmp2; gc.collect()
    tmp = tmp.reset_index().sort_values(['aid_x','wgt'], ascending=[True, False])
    tmp = tmp.reset_index(drop=True)
    tmp['n'] = tmp.groupby('aid_x').aid_y.cumcount()
    tmp = tmp.loc[tmp['n']<20].drop(columns=['n'])
    tmp.to_pandas().to_parquet(f'top_20_clicks_v{VER}_{PART}.pqt')
print('clicks done.')


In [ ]:
del data_cache; gc.collect()

## Step 2 — Load test_df & top popular in test (numeric types)

In [ ]:
def load_test():
    dfs = []
    for chunk_file in sorted(glob.glob('../input/otto-chunk-data-inparquet-format/test_parquet/*')):
        chunk = pd.read_parquet(chunk_file)
        if chunk['ts'].max()>1e10:
            chunk['ts'] = (chunk['ts']/1000).astype('int32')
        else:
            chunk['ts'] = chunk['ts'].astype('int32')
        if chunk['type'].dtype=='O':
            chunk['type'] = chunk['type'].map(type_labels).astype('int8')
        dfs.append(chunk)
    return pd.concat(dfs).reset_index(drop=True)

test_df = load_test()
print('Test shape:', test_df.shape)

# numeric type (0/1/2) — correct!
top_clicks = test_df.loc[test_df['type']==0,'aid'].value_counts().index.values[:20]
top_orders = test_df.loc[test_df['type']==2,'aid'].value_counts().index.values[:20]
top_clicks_set, top_orders_set = set(top_clicks), set(top_orders)
print('Top populars ready:', len(top_clicks), len(top_orders))


## Step 3 — Read co-vis parquets into **weights** & **ranks** dicts

In [ ]:
def pqt_to_weighted_and_rank(paths):
    weights, ranks = defaultdict(dict), defaultdict(dict)
    for p in paths:
        df = pd.read_parquet(p, columns=['aid_x','aid_y','wgt'])
        df = df.sort_values(['aid_x','wgt'], ascending=[True, False])
        df['rank'] = df.groupby('aid_x').cumcount().astype('int32')
        for ax, ay, w, r in zip(df['aid_x'].values, df['aid_y'].values, df['wgt'].values, df['rank'].values):
            ax, ay = int(ax), int(ay)
            weights[ax][ay] = float(w)
            ranks[ax][ay]   = int(r)
    return dict(weights), dict(ranks)

CLICK_PQTS = sorted(glob.glob('top_20_clicks_v*_*.pqt'))
BUYS_PQTS  = sorted(glob.glob('top_15_carts_orders_v*_*.pqt'))
B2B_PQTS   = sorted(glob.glob('top_15_buy2buy_v*_*.pqt'))
print('Found pqts:', len(CLICK_PQTS), len(BUYS_PQTS), len(B2B_PQTS))

click_w, click_r = pqt_to_weighted_and_rank(CLICK_PQTS)
buys_w,  buys_r  = pqt_to_weighted_and_rank(BUYS_PQTS)
b2b_w,   b2b_r   = pqt_to_weighted_and_rank(B2B_PQTS)


## Step 4 — Handcrafted RULE suggestors (baseline)

In [ ]:
TYPE_W = {0:1, 1:6, 2:3}

def suggest_clicks(df):
    aids = df.aid.tolist(); types = df.type.tolist()
    unique_aids = list(dict.fromkeys(aids[::-1]))
    if len(unique_aids) >= 20:
        w = np.logspace(0.1, 1, len(aids), base=2, endpoint=True) - 1
        sc = Counter()
        for a, ww, t in zip(aids, w, types):
            sc[a] += float(ww) * TYPE_W[int(t)]
        return [k for k,_ in sc.most_common(20)]
    aids2 = list(itertools.chain(*[click_w.get(a,{}).keys() for a in unique_aids]))
    top2 = [aid for aid,_ in Counter(aids2).most_common(20) if aid not in unique_aids]
    res = unique_aids + top2[:20-len(unique_aids)]
    return res + list(top_clicks)[:20-len(res)]

def suggest_buys(df):
    aids = df.aid.tolist(); types = df.type.tolist()
    unique_aids = list(dict.fromkeys(aids[::-1]))
    dfb = df[df['type'].isin([1,2])]
    unique_buys = list(dict.fromkeys(dfb.aid.tolist()[::-1]))
    if len(unique_aids) >= 20:
        w = np.logspace(0.5, 1, len(aids), base=2, endpoint=True) - 1
        sc = Counter()
        for a, ww, t in zip(aids, w, types):
            sc[a] += float(ww) * TYPE_W[int(t)]
        # add buy2buy light boost
        aids3 = list(itertools.chain(*[b2b_w.get(a,{}).keys() for a in unique_buys]))
        for a in aids3: sc[a] += 0.1
        return [k for k,_ in sc.most_common(20)]
    aids2 = list(itertools.chain(*[buys_w.get(a,{}).keys()  for a in unique_aids]))
    aids3 = list(itertools.chain(*[b2b_w.get(a,{}).keys()   for a in unique_buys]))
    top2 = [aid for aid,_ in Counter(aids2+aids3).most_common(20) if aid not in unique_aids]
    res = unique_aids + top2[:20-len(unique_aids)]
    return res + list(top_orders)[:20-len(res)]


## Step 5 — Candidates & Features (with **correct w_b2b**)

In [ ]:
def build_session_seeds(df_hist):
    aids  = df_hist.aid.tolist()
    types = df_hist.type.tolist()
    unique_aids = list(dict.fromkeys(aids[::-1]))
    unique_buys = list(dict.fromkeys(df_hist[df_hist['type'].isin([1,2])].aid.tolist()[::-1]))
    return aids, types, unique_aids, unique_buys

def recent_type_score(aids, types, lo=0.1, hi=1.0):
    w = np.logspace(lo, hi, len(aids), base=2, endpoint=True) - 1.0
    sc = Counter()
    for a, ww, t in zip(aids, w, types):
        sc[a] += float(ww) * TYPE_W[int(t)]
    return sc

def aggregate_neighbors(unique_aids, unique_buys):
    agg_click, agg_buys, agg_b2b = Counter(), Counter(), Counter()
    for a in unique_aids:
        for b,w in click_w.get(a, {}).items(): agg_click[b] += w
        for b,w in buys_w.get(a,  {}).items(): agg_buys[b]  += w
    for a in unique_buys:
        for b,w in b2b_w.get(a,   {}).items(): agg_b2b[b]   += w
    return agg_click, agg_buys, agg_b2b

def build_candidates(df_hist, k_click=120, k_buys=120):
    aids, types, unique_aids, unique_buys = build_session_seeds(df_hist)
    agg_click, agg_buys, agg_b2b = aggregate_neighbors(unique_aids, unique_buys)
    cand_clicks = set(unique_aids) | set([b for b,_ in agg_click.most_common(k_click)])
    cand_buys   = set(unique_aids) | set([b for b,_ in (agg_buys+agg_b2b).most_common(k_buys)])
    return (aids, types, unique_aids, unique_buys, agg_click, agg_buys, agg_b2b, cand_clicks, cand_buys)

def best_rank_over_seeds(c, seeds, ranks, default=9999):
    best = default
    for s in seeds:
        r = ranks.get(s, {}).get(c, default)
        if r < best: best = r
    return best

def make_features_for_head(df_hist, head, candidates,
                           aids, types, unique_aids, unique_buys,
                           agg_click, agg_buys, agg_b2b):
    rec_sc  = recent_type_score(aids, types)
    sesslen = len(aids)
    last_ts = int(df_hist.ts.max()) if sesslen else 0
    rows, keys = [], []
    for c in candidates:
        in_hist = 1 if c in rec_sc else 0
        w_click = float(agg_click.get(c, 0.0))
        w_buys  = float(agg_buys.get(c, 0.0))
        w_b2b   = float(agg_b2b.get(c, 0.0))  # FIXED: correct source
        r_click = best_rank_over_seeds(c, unique_aids, click_r)
        r_buys  = best_rank_over_seeds(c, unique_aids, buys_r)
        r_b2b   = best_rank_over_seeds(c, unique_buys, b2b_r)
        f_best_click = 1.0/(1.0 + r_click)
        f_best_buys  = 1.0/(1.0 + r_buys)
        f_best_b2b   = 1.0/(1.0 + r_b2b)
        f_rec_sc = float(rec_sc.get(c, 0.0))
        if c in df_hist.aid.values:
            gap = last_ts - int(df_hist[df_hist.aid==c].ts.max())
        else:
            gap = 10*24*3600
        # stabilize gap
        gap = np.log1p(min(gap, 3*24*3600))
        is_top_click = int(c in top_clicks_set)
        is_top_order = int(c in top_orders_set)
        if head=='clicks':
            rows.append([in_hist, f_rec_sc, w_click, f_best_click,
                         w_buys, f_best_buys, f_best_b2b,
                         sesslen, gap, is_top_click, is_top_order])
        else:
            rows.append([in_hist, f_rec_sc, w_buys, f_best_buys,
                         w_click, f_best_click, f_best_b2b,
                         sesslen, gap, is_top_click, is_top_order])
        keys.append(c)
    X = pd.DataFrame(rows, columns=[
        'in_hist','rec_rule',
        'w_main','best_main',
        'w_aux1','best_aux1',
        'best_b2b',
        'sess_len','gap',
        'is_top_click','is_top_order'
    ])
    for c in X.columns:
        X[c] = X[c].astype('float32') if c not in ('in_hist','is_top_click','is_top_order') else X[c].astype('int8')
    return X, keys


## Step 6 — Install **xgboost** & train **XGBRanker(group)**

In [ ]:
!pip -q install xgboost==1.6.2
import xgboost as xgb
from xgboost.sklearn import XGBRanker
print('xgboost version:', xgb.__version__)

def build_train_rows_from_session(df_sess):
    if df_sess.empty: return None
    split_ts = int(df_sess.ts.max()) - 24*60*60
    hist = df_sess[df_sess.ts <  split_ts]
    fut  = df_sess[df_sess.ts >= split_ts]
    if hist.empty or fut.empty: return None
    (aids, types, unique_aids, unique_buys,
     agg_click, agg_buys, agg_b2b,
     cand_clicks, cand_buys) = build_candidates(hist)
    labs_clicks = set(fut.loc[fut['type']==0,'aid'].tolist())
    labs_buys   = set(fut.loc[fut['type'].isin([1,2]),'aid'].tolist())
    Xc, kc = make_features_for_head(hist, 'clicks', cand_clicks,
                                    aids, types, unique_aids, unique_buys,
                                    agg_click, agg_buys, agg_b2b)
    yc = np.array([1 if a in labs_clicks else 0 for a in kc], dtype=np.int32)
    Xb, kb = make_features_for_head(hist, 'buys', cand_buys,
                                    aids, types, unique_aids, unique_buys,
                                    agg_click, agg_buys, agg_b2b)
    yb = np.array([1 if a in labs_buys else 0 for a in kb], dtype=np.int32)
    return (Xc, yc), (Xb, yb)

def build_training_dataset(train_glob="../input/otto-chunk-data-inparquet-format/*_parquet/*",
                           max_sessions=80000, seed=42):
    files = glob.glob(train_glob)
    random.seed(seed); random.shuffle(files)
    Xc_list, yc_list, Xb_list, yb_list = [], [], [], []
    seen = 0
    for f in files:
        df = pd.read_parquet(f)
        if df['ts'].max() > 1e10:
            df['ts'] = (df['ts']/1000).astype('int32')
        else:
            df['ts'] = df['ts'].astype('int32')
        if df['type'].dtype=='O':
            df['type'] = df['type'].map(type_labels).astype('int8')
        for _, g in df.groupby('session'):
            out = build_train_rows_from_session(g)
            if out is None:
                continue
            (Xc,yc),(Xb,yb) = out
            if not Xc.empty:
                Xc_list.append(Xc); yc_list.append(yc)
            if not Xb.empty:
                Xb_list.append(Xb); yb_list.append(yb)
            seen += 1
            if seen >= max_sessions:
                break
        if seen >= max_sessions:
            break
    def _concat_and_groups(X_list, y_list):
        groups = [len(df) for df in X_list]
        X = pd.concat(X_list, ignore_index=True) if X_list else pd.DataFrame()
        y = np.concatenate(y_list) if y_list else np.array([], np.int32)
        return X, y, groups
    Xc, yc, groups_c = _concat_and_groups(Xc_list, yc_list)
    Xb, yb, groups_b = _concat_and_groups(Xb_list, yb_list)
    return Xc, yc, groups_c, Xb, yb, groups_b

Xc, yc, groups_c, Xb, yb, groups_b = build_training_dataset(max_sessions=100000)
print('Clicks train:', Xc.shape, 'pos rate:', float(yc.mean()) if len(yc) else None, 'groups:', len(groups_c))
print('Buys   train:', Xb.shape, 'pos rate:', float(yb.mean()) if len(yb) else None, 'groups:', len(groups_b))

ranker_clicks = XGBRanker(
    objective='rank:pairwise',
    n_estimators=350, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=1e-3, reg_lambda=1.0,
    tree_method='gpu_hist', predictor='gpu_predictor',
    random_state=2025, n_jobs=-1
)
ranker_buys = XGBRanker(
    objective='rank:pairwise',
    n_estimators=400, max_depth=7, learning_rate=0.05,
    subsample=0.85, colsample_bytree=0.85,
    reg_alpha=1e-3, reg_lambda=1.0,
    tree_method='gpu_hist', predictor='gpu_predictor',
    random_state=2025, n_jobs=-1
)

if (not Xc.empty) and (len(np.unique(yc))>1):
    ranker_clicks.fit(Xc.values, yc, group=groups_c, verbose=False)
if (not Xb.empty) and (len(np.unique(yb))>1):
    ranker_buys.fit(Xb.values, yb, group=groups_b, verbose=False)


## Step 7 — Rank & BLEND helpers

In [ ]:
def rank_and_take20(model, X, keys, fallback_sorted, rule_score=None, alpha=None):
    if X is None or (hasattr(X, 'empty') and X.empty):
        return fallback_sorted[:20]
    use_model = (model is not None) and hasattr(model, 'predict')
    s_m = model.predict(X.values).astype(float) if use_model else np.zeros(len(X), dtype=float)
    if alpha is not None and rule_score is not None:
        s = alpha * s_m + (1.0 - alpha) * np.asarray(rule_score, dtype=float)
    else:
        s = s_m
    order = np.argsort(-s)
    ranked = [keys[i] for i in order]
    out, used = [], set()
    for a in ranked + fallback_sorted:
        if a not in used:
            out.append(a); used.add(a)
        if len(out)==20: break
    return out

def suggest_clicks_ml(df, alpha=None):
    (aids, types, unique_aids, unique_buys,
     agg_click, agg_buys, agg_b2b,
     cand_clicks, _) = build_candidates(df)
    Xc, kc = make_features_for_head(df, 'clicks', cand_clicks,
                                    aids, types, unique_aids, unique_buys,
                                    agg_click, agg_buys, agg_b2b)
    fallback = suggest_clicks(df)
    rule_c = (Xc['rec_rule'] + 0.8*Xc['w_main'] + 0.2*Xc['best_main']).values if (alpha is not None) else None
    return rank_and_take20(ranker_clicks if 'ranker_clicks' in globals() else None,
                           Xc, kc, fallback, rule_score=rule_c, alpha=alpha)

def suggest_buys_ml(df, alpha=None):
    (aids, types, unique_aids, unique_buys,
     agg_click, agg_buys, agg_b2b,
     _, cand_buys) = build_candidates(df)
    Xb, kb = make_features_for_head(df, 'buys', cand_buys,
                                    aids, types, unique_aids, unique_buys,
                                    agg_click, agg_buys, agg_b2b)
    fallback = suggest_buys(df)
    rule_b = (Xb['rec_rule'] + 0.9*Xb['w_main'] + 0.2*Xb['best_main']).values if (alpha is not None) else None
    return rank_and_take20(ranker_buys if 'ranker_buys' in globals() else None,
                           Xb, kb, fallback, rule_score=rule_b, alpha=alpha)


## Step 8 — Batch inference (one pass per head, optional BLEND)

In [ ]:
import time
try:
    import psutil
except Exception:
    psutil = None

def _rank_take20_with_fallback(scores_keys_per_sess, fallback_per_sess):
    out = {}
    for sid, pairs in scores_keys_per_sess.items():
        pairs.sort(key=lambda x: -x[0])
        used, res = set(), []
        for sc, aid in pairs:
            if aid not in used:
                res.append(aid); used.add(aid)
            if len(res)==20: break
        if len(res)<20:
            for aid in fallback_per_sess.get(sid, []):
                if aid not in used:
                    res.append(aid); used.add(aid)
                if len(res)==20: break
        out[sid] = res
    return out

def _batch_predict_head(head, df_sorted, batch_sessions=120000, alpha=None):
    assert head in ('clicks','buys')
    model = ranker_clicks if head=='clicks' else ranker_buys
    use_model = (model is not None) and hasattr(model, 'predict')
    preds_all = {}
    sessions = df_sorted['session'].drop_duplicates().values
    N = len(sessions)
    print(f"\nStart rerank for [{head}] on {N:,} sessions; batch={batch_sessions}")
    t0_total = time.time()
    for start in range(0, N, batch_sessions):
        end = min(N, start+batch_sessions)
        sess_batch = set(sessions[start:end])
        batch_id = start//batch_sessions + 1
        print(f"\nBatch {batch_id}: {start:,}~{end-1:,} ({end-start:,})")
        g = df_sorted[df_sorted['session'].isin(sess_batch)].groupby('session', sort=False)
        X_rows, key_rows, rule_rows, fallback = [], [], [], {}
        for sid, df_sess in g:
            (aids, types, unique_aids, unique_buys,
             agg_click, agg_buys, agg_b2b,
             cand_clicks, cand_buys) = build_candidates(df_sess)
            if head=='clicks':
                X, keys = make_features_for_head(df_sess, 'clicks', cand_clicks,
                                                 aids, types, unique_aids, unique_buys,
                                                 agg_click, agg_buys, agg_b2b)
                fallback[sid] = suggest_clicks(df_sess)
                rule = (X['rec_rule'] + 0.8*X['w_main'] + 0.2*X['best_main']).values if (alpha is not None) else None
            else:
                X, keys = make_features_for_head(df_sess, 'buys', cand_buys,
                                                 aids, types, unique_aids, unique_buys,
                                                 agg_click, agg_buys, agg_b2b)
                fallback[sid] = suggest_buys(df_sess)
                rule = (X['rec_rule'] + 0.9*X['w_main'] + 0.2*X['best_main']).values if (alpha is not None) else None
            if not X.empty:
                X_rows.append(X)
                key_rows.extend([(sid, a) for a in keys])
                if rule is not None: rule_rows.append(rule)
        if not X_rows:
            print('No features in batch. Skipping.')
            continue
        X_all = pd.concat(X_rows, ignore_index=True)
        rule_all = np.concatenate(rule_rows) if rule_rows else None
        if use_model:
            s_m = model.predict(X_all.values).astype(float)
        else:
            s_m = np.zeros(len(X_all), dtype=float)
        scores = s_m if (alpha is None or rule_all is None) else (alpha*s_m + (1.0-alpha)*rule_all)
        scores_per_sess = defaultdict(list)
        for (sid, aid), sc in zip(key_rows, scores):
            scores_per_sess[sid].append((float(sc), int(aid)))
        batch_pred = _rank_take20_with_fallback(scores_per_sess, fallback)
        preds_all.update(batch_pred)
        if psutil:
            print(f"Done batch {batch_id} | mem used ~ {psutil.virtual_memory().used/1024**3:.2f} GB")
        del X_rows, key_rows, rule_rows, X_all, scores_per_sess, batch_pred
    print(f"Finish [{head}] in {time.time()-t0_total:.1f}s")
    return preds_all


## Step 9 — Generate submissions for RULE / RANK / BLEND (one-shot)

In [ ]:
def _to_submission_block(d, suffix):
    s = pd.Series({f"{sid}_{suffix}": " ".join(map(str, aids)) for sid, aids in d.items()})
    return s.rename_axis('session_type').reset_index(name='labels')

test_sorted = test_df.sort_values(['session','ts'], kind='mergesort')

# RULE
clicks_rule = {}
buys_rule   = {}
for sid, g in test_sorted.groupby('session'):
    clicks_rule[int(sid)] = suggest_clicks(g)
    buys_rule[int(sid)]   = suggest_buys(g)
sub_rule = pd.concat([
    _to_submission_block(clicks_rule, 'clicks'),
    _to_submission_block(buys_rule,   'orders'),
    _to_submission_block(buys_rule,   'carts')
], ignore_index=True)
sub_rule.to_csv('submission_rules.csv', index=False)

# RANK (alpha=None)
pred_clicks_rank = _batch_predict_head('clicks', test_sorted, batch_sessions=120000, alpha=None)
pred_buys_rank   = _batch_predict_head('buys',   test_sorted, batch_sessions=120000, alpha=None)
sub_rank = pd.concat([
    _to_submission_block(pred_clicks_rank, 'clicks'),
    _to_submission_block(pred_buys_rank,   'orders'),
    _to_submission_block(pred_buys_rank,   'carts')
], ignore_index=True)
sub_rank.to_csv('submission_rank.csv', index=False)

# BLEND (alpha=0.35)
pred_clicks_blend = _batch_predict_head('clicks', test_sorted, batch_sessions=120000, alpha=0.35)
pred_buys_blend   = _batch_predict_head('buys',   test_sorted, batch_sessions=120000, alpha=0.35)
sub_blend = pd.concat([
    _to_submission_block(pred_clicks_blend, 'clicks'),
    _to_submission_block(pred_buys_blend,   'orders'),
    _to_submission_block(pred_buys_blend,   'carts')
], ignore_index=True)
sub_blend.to_csv('submission_blend.csv', index=False)

print('Saved: submission_rules.csv, submission_rank.csv, submission_blend.csv')


## Step 10 — CV (Radek split, cuDF evaluation) for RULE / RANK / BLEND

In [ ]:
import cupy as cp
CUT_TS = int(test_df['ts'].max()) - 24*60*60
gdf = cudf.from_pandas(test_df[['session','aid','ts','type']].copy())
gdf['is_click'] = (gdf['type']==0).astype('int8')
gdf['is_buy']   = gdf['type'].isin([1,2]).astype('int8')
sessions_g = gdf['session'].unique()

def _explode_preds(preds):
    rows = []
    for sid, arr in preds.items():
        for a in arr[:20]:
            rows.append((sid, int(a)))
    return cudf.DataFrame(rows, columns=['session','aid'])

def _recall_mean(pred_df, truth_df):
    hit = pred_df.merge(truth_df, on=['session','aid'], how='inner')\
                 .groupby('session').size().rename('hit')
    gt  = truth_df.groupby('session').size().rename('gt')
    stat = cudf.DataFrame({'session': sessions_g}).merge(gt.reset_index(), on='session', how='left')\
                                                .merge(hit.reset_index(), on='session', how='left')\
                                                .fillna({'gt':0,'hit':0})
    denom = cp.minimum(20, stat['gt'])
    rec = (stat['hit'] / denom.replace(0, cp.nan)).fillna(0.0)
    return float(rec.to_pandas().mean())

def _eval_mode(mode_name, clicks_pred, buys_pred):
    predC = _explode_preds(clicks_pred)
    predB = _explode_preds(buys_pred)
    futC = gdf[(gdf['ts']>=CUT_TS) & (gdf['is_click']==1)][['session','aid']].drop_duplicates()
    futB = gdf[(gdf['ts']>=CUT_TS) & (gdf['is_buy'  ]==1)][['session','aid']].drop_duplicates()
    Rc = _recall_mean(predC, futC)
    Rb = _recall_mean(predB, futB)
    score = 0.10*Rc + 0.90*Rb
    print(f"[{mode_name}] clicks_R={Rc:.5f}  buys_R={Rb:.5f}  ==> score={score:.5f}")
    return Rc, Rb, score

# Build per-session predictions for the three modes on HIST/FUT split
clicks_RULE_CV, buys_RULE_CV = {}, {}
clicks_RANK_CV, buys_RANK_CV = {}, {}
clicks_BLEND_CV, buys_BLEND_CV = {}, {}

for sid, gs in test_df.groupby('session'):
    hist = gs[gs['ts'] < CUT_TS]
    fut  = gs[gs['ts'] >= CUT_TS]
    if hist.empty or fut.empty: 
        continue
    clicks_RULE_CV[sid] = suggest_clicks(hist)
    buys_RULE_CV[sid]   = suggest_buys(hist)
    clicks_RANK_CV[sid] = suggest_clicks_ml(hist, alpha=None)
    buys_RANK_CV[sid]   = suggest_buys_ml(hist,   alpha=None)
    clicks_BLEND_CV[sid] = suggest_clicks_ml(hist, alpha=0.35)
    buys_BLEND_CV[sid]   = suggest_buys_ml(hist,   alpha=0.35)

_eval_mode('RULE', clicks_RULE_CV, buys_RULE_CV)
_eval_mode('RANK', clicks_RANK_CV, buys_RANK_CV)
_eval_mode('BLEND-0.35', clicks_BLEND_CV, buys_BLEND_CV)
